# Воспроизведение решения на Kaggle

Этот ноутбук одинаково работает в ветках `spatial-ensemble` и `prototype-baseline`. Он получает признаки из исходных фотографий и применяет приложенные обученные модели.

1. Скачайте ZIP **нужной ветки** GitHub и добавьте его в приватный Kaggle Dataset. Подключите этот Dataset к ноутбуку.
2. В настройках Kaggle включите NVIDIA GPU и Internet. Установщик сам подготовит Python 3.11, даже если Python ядра другой.
3. Для SAM3.1 и DINOv3 нужен ранее полученный доступ Hugging Face. Подключите свой `HF_TOKEN` через Add-ons → Secrets. Токен не нужно вставлять в код.
4. Выполните ячейки по порядку. Все команды вызывают скрипты из выбранной ветки.

Полный пересчёт может давать численный дрейф относительно исходных MPS/CUDA-экспериментов. Для точного опубликованного CSV используйте короткий CPU-маршрут из README. Статус фактически выполненных проверок — в `REPRODUCIBILITY.md`.


In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil

SOURCE = None  # При нескольких архивах: Path('/kaggle/input/.../папка-репозитория')
ROOT = Path('/kaggle/temp/rascar-reproduction')
OUT = Path('/kaggle/working/rascar-results')
RAW = ROOT/'cache/competition'
START = 'metadata'  # Для продолжения: foundation, local, guards, tta, body, sahi, kernel, boundary, final
RETRAIN = False    # True = обучить всю цепочку; нужен новый, ещё не существующий OUT

if SOURCE is None:
    candidates = [p.parent for p in Path('/kaggle/input').rglob('from_images.py')
                  if (p.parent/'checksums.json').is_file()]
    if len(candidates) != 1:
        raise RuntimeError(f'Подключите ZIP одной ветки или укажите SOURCE. Найдены: {candidates}')
    SOURCE = candidates[0]
assert (SOURCE/'from_images.py').is_file(), SOURCE
shutil.copytree(SOURCE, ROOT, dirs_exist_ok=True)
os.chdir(ROOT)
env = dict(os.environ, UV_CACHE_DIR='/kaggle/temp/uv-cache',
           UV_PYTHON_INSTALL_DIR='/kaggle/temp/python', HF_HOME='/kaggle/temp/huggingface',
           OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')
print('Репозиторий:', SOURCE, 'Результаты:', OUT)
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
# uv хранит общие пакеты один раз; окружения остаются изолированными.
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True, env=env)
subprocess.run([sys.executable, '-m', 'uv', 'python', 'install', '3.11'], check=True, env=env)
py311 = subprocess.check_output(
    [sys.executable, '-m', 'uv', 'python', 'find', '3.11'], text=True, env=env).strip()
subprocess.run([py311, str(ROOT/'setup_env.py'), '--gpu', '--installer', 'uv'],
               cwd=ROOT, env=env, check=True)
PY = str(ROOT/'.venv/bin/python')
subprocess.run([PY, '-m', 'pip', 'install', '-r', str(ROOT/'requirements-download.txt')],
               cwd=ROOT, env=env, check=True)


In [ ]:
# Используется уже подключённый секрет; разрешения и условия доступа не меняются.
if not env.get('HF_TOKEN'):
    from kaggle_secrets import UserSecretsClient
    try:
        env['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        raise RuntimeError('Подключите действующий HF_TOKEN через Add-ons → Secrets. '
                           'Аккаунту нужен доступ к facebook/sam3.1 и facebook/dinov3-vitb16-pretrain-lvd1689m.') from None

subprocess.run([PY, str(ROOT/'download_data.py'), '--out', str(RAW)], cwd=ROOT, env=env, check=True)
subprocess.run([PY, str(ROOT/'download_weights.py')], cwd=ROOT, env=env, check=True)


In [ ]:
cmd = [PY, str(ROOT/'from_images.py'), '--raw', str(RAW), '--out', str(OUT),
       '--device', 'cuda:0', '--start', START]
if RETRAIN:
    cmd.append('--retrain')
subprocess.run(cmd, cwd=ROOT, env=env, check=True)
subprocess.run([PY, str(ROOT/'verify.py'), '--out', str(OUT/'predictions'), '--allow-drift'],
               cwd=ROOT, env=env, check=True)


In [ ]:
from IPython.display import FileLink, display
submission = OUT/'predictions/submission.csv'
assert submission.is_file(), submission
print('Готово:', submission)
print('Сравнение с эталоном находится в выводе verify.py выше.')
os.chdir('/kaggle/working')
display(FileLink(str(submission.relative_to('/kaggle/working'))))
